# E6 | Model Workload Forecast — Por Equipe
Prever volume de incidentes por equipe (Forecast D+1 a D+7 segmentado)

## 📋 Objetivo

Neste notebook, vou treinar modelos **Prophet independentes** para cada equipe (grupo_designado) afim de responder:
**\"Quais equipes podem ser sobrecarregadas na próxima semana?\"**

### Por que separar por equipe?
- O modelo anterior (05) prevê **volume total** de incidentes (útil para infraestrutura geral)
- Este modelo prevê **volume por equipe** (útil para alocação de recursos)
- Equipes têm padrões diferentes:
  - **Team1**: Mais incidentes às segundas-feiras (sistema de batch)
  - **Team14**: Mais incidentes às sextas-feiras (preparação do fim de semana)
  - **Team3**: Padrão mais regular

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import os, pandas as pd, numpy as np
from datetime import datetime, timedelta
from sqlalchemy import create_engine
from dotenv import load_dotenv
from prophet import Prophet
from sklearn.metrics import mean_absolute_percentage_error
import mlflow

load_dotenv()
print('✅ Setup OK')

In [ ]:
RDS_HOST     = os.getenv('RDS_HOST')
RDS_PORT     = int(os.getenv('RDS_PORT', '5432'))
RDS_USER     = os.getenv('RDS_USER', 'postgres')
RDS_PASSWORD = os.getenv('RDS_PASSWORD')
RDS_DATABASE = os.getenv('RDS_DATABASE', 'aiops_gold')

engine = create_engine(f\"postgresql://{RDS_USER}:{RDS_PASSWORD}@{RDS_HOST}:{RDS_PORT}/{RDS_DATABASE}\")

query = '''
    SELECT DATE(data_abertura) AS data_abertura,
           grupo_designado,
           COUNT(*) AS total_chamados
    FROM gold_ml.ml_forecast_dataset
    WHERE grupo_designado IS NOT NULL
    GROUP BY DATE(data_abertura), grupo_designado
    ORDER BY data_abertura, grupo_designado
'''
df = pd.read_sql(query, engine)
df['data_abertura'] = pd.to_datetime(df['data_abertura'])

print(f'✅ Carregados {len(df)} registros diários por equipe')
print(f'   Equipes únicas: {df.grupo_designado.nunique()}')

In [ ]:
mlflow.set_experiment('prophet_workload_by_team')

teams = df['grupo_designado'].unique()
models_by_team = {}
team_metrics = []

print(f'🚀 Treinando {len(teams)} modelos Prophet (um por equipe)...')

for team in sorted(teams):
    df_team = df[df['grupo_designado'] == team].copy()
    df_team = df_team.rename(columns={'data_abertura': 'ds', 'total_chamados': 'y'})
    df_team = df_team[['ds', 'y']].sort_values('ds').reset_index(drop=True)
    
    if len(df_team) < 60:
        continue
    
    with mlflow.start_run(nested=True, run_name=f'team_{team}'):
        train = df_team.iloc[:-30]
        test = df_team.iloc[-30:]
        
        model = Prophet(yearly_seasonality=False, weekly_seasonality=True, seasonality_mode='additive')
        model.fit(train)
        
        forecast = model.predict(test[['ds']])
        mape = mean_absolute_percentage_error(test['y'].values, forecast['yhat'].values)
        
        models_by_team[team] = model
        team_metrics.append({'team': team, 'n_days': len(df_team), 'mape': mape})
        
        mlflow.log_params({'team': team, 'n_days': len(df_team)})
        mlflow.log_metrics({'mape': float(mape)})
        mlflow.set_tag('model_type', 'Prophet-Team-Forecast')
        
        print(f'  ✅ {team}: MAPE={mape:.2%}')

print(f'\n🏆 {len(models_by_team)} equipes treinadas com sucesso')

In [ ]:
from pathlib import Path

base_path = Path(r'D:\Projetos\AWS_Portifolio\Projeto aws\data\ml\forecast_by_team')
base_path.mkdir(parents=True, exist_ok=True)

df_metrics = pd.DataFrame(team_metrics)
df_metrics.to_csv(str(base_path / 'forecast_by_team_metrics.csv'), index=False)

print(f'✅ Salvos em {base_path}')
print(f'📊 Resumo:')
print(df_metrics.to_string(index=False))